# Cross-band speckle statistics in code

[The cross-band statistics page](../explanation/cross-band-statistics)
derives the joint law that governs a chromatic `SpeckleProcess`: every
wavelength channel is driven by the same real mode vector, so the channels
are one improper complex Gaussian field over pixel and wavelength rather
than independent draws. This notebook builds a small chromatic process and
drives `cross_band_moments`, its derived views, `joint_covariance`, and the
`scale_e_nom` fix end to end, checking the numbers against the closed forms
as it goes.

Notation follows the theory page: $x = (r, \lambda)$ for a pixel-wavelength
pair, $\Gamma_{ij}$ and $P_{ij}$ for the covariance and pseudo-covariance
kernels, and $\rho_{ij}$ for the band-pair correlation.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)  # deep contrast and small rho gaps need it

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from physicaloptix import (
    Field,
    Fraunhofer,
    Grid,
    OpticalPath,
    PlaneKind,
    SpeckleProcess,
    Spectrum,
    Stage,
    broadcast_to_spectrum,
    fourier_dm_basis,
    lambda_scaled_channels,
    linearize,
    stats,
)
from physicaloptix.coatings import thickness_kernel

NPUP, NFOC, PSCALE = 48, 96, 0.25  # small grids so the notebook runs quickly
WLS = jnp.linspace(480.0, 650.0, 8)  # 8 channels across a wide band
WL0 = float(WLS[0])

pupil = Grid.pupil(NPUP)
focal = Grid.focal(NFOC, PSCALE)
x = np.asarray(pupil.coords)
xg, yg = np.meshgrid(x, x)
aperture = ((xg**2 + yg**2) <= 0.25).astype(complex)
flat = Field(data=jnp.asarray(aperture), grid=pupil, plane=PlaneKind.PUPIL)

spectrum = Spectrum(wavelengths_nm=WLS, weights=jnp.full(WLS.shape, 1.0 / WLS.shape[0]))
chromatic_field = broadcast_to_spectrum(flat, spectrum)
path = OpticalPath(stages=(Stage("sci", Fraunhofer(grid_in=pupil, grid_out=focal)),))

# a band-limited control basis over the 2-4 lambda/D annulus
basis = fourier_dm_basis(pupil, n_actuators=8, k_min=2.0, k_max=4.0, rms_nm=1.0)
print(f"basis modes: {basis.n_modes}, wavelengths: {np.asarray(WLS)}")

## The achromatic baseline

`linearize` accepts `wavelengths_nm` directly and builds the full per-band
stack in one call: `e_nom` becomes `(w, y, x)` and `G` becomes `(w, m, y, x)`,
with the default per-mode dispersion `i 2 pi / lambda` (the OPD law) at every
wavelength. Because the telescope here is achromatic and the focal grid stays
in native $\lambda/D$ units, this is exactly the achromatic limit the theory
page anchors on: incoherent statistics should give $\rho_{ij} \equiv 1$ and
$N_\mathrm{eff} \equiv 1$ everywhere, in any basis.

In [ ]:
lin = linearize(path, chromatic_field, basis, wavelengths_nm=WLS)
print(f"e_nom {lin.e_nom.shape}, G {lin.G.shape}")

proc = lin.to_speckle_process(decorr_hours=2.0, total_rms=5.0, coherent=False)
cbm = proc.cross_band_moments()
rho = np.asarray(cbm.correlation())
n_eff = np.asarray(cbm.n_eff())
print(f"rho range:   [{rho.min():.9f}, {rho.max():.9f}]")
print(f"n_eff range: [{n_eff.min():.9f}, {n_eff.max():.9f}]")

mask = np.asarray(stats.dark_zone_mask(focal, iwa_lod=2.0, owa_lod=4.0))
print(f"dark-zone pixels in the controlled annulus: {mask.sum()}")

## A genuinely chromatic element

A single shared dispersion curve, however it is shaped, cannot decorrelate
bands: every mode would still scale as one common $c(\lambda)$, and the
achromatic argument above goes through unchanged regardless of what
$c(\lambda)$ looks like. Genuine decorrelation needs two DIFFERENT dispersion
curves mixed across the mode set, which is what happens physically when a
beam sees a coating: some spatial content responds through the plain
OPD-to-phase law, some through the coating's own group delay. `linearize`
exposes exactly this through its `dispersion` keyword, a `(m, w)` table of
per-mode complex factors that overrides the default. We build one here: half
the modes keep the OPD law, and half take the derivative of a small quarter-wave
coating stack's reflectance phase with respect to layer thickness
(`coatings.thickness_kernel`), rescaled to a comparable magnitude so neither
channel trivially dominates.

In [ ]:
m = basis.n_modes
n_h, n_l = 2.35, 1.45  # a simple high/low index pair (titania/silica-like)
d_h, d_l = 550.0 / (4.0 * n_h), 550.0 / (4.0 * n_l)  # quarter-wave at 550 nm
indices = [n_h, n_l] * 4
thicknesses = [d_h, d_l] * 4
coat = thickness_kernel(WLS, indices, thicknesses, 0, output="r", n_substrate=1.50)
opd_factor = 1j * 2.0 * jnp.pi / WLS
scale = float(jnp.abs(opd_factor).mean() / jnp.abs(coat).mean())

half = m // 2
dispersion = jnp.zeros((m, WLS.shape[0]), dtype=complex)
dispersion = dispersion.at[:half].set(
    jnp.broadcast_to(opd_factor, (half, WLS.shape[0]))
)
dispersion = dispersion.at[half:].set(
    jnp.broadcast_to(coat * scale, (m - half, WLS.shape[0]))
)

lin2 = linearize(
    path, chromatic_field, basis, wavelengths_nm=WLS, dispersion=dispersion
)
proc2 = lin2.to_speckle_process(decorr_hours=2.0, total_rms=5.0, coherent=False)
cbm2 = proc2.cross_band_moments()
rho2 = np.asarray(cbm2.correlation())

ys, xs = np.nonzero(mask)
py, px = ys[len(ys) // 2], xs[len(ys) // 2]  # one representative dark-zone pixel
print(f"sample pixel {(py, px)}, rho(lam_0, lam):\n{rho2[0, :, py, px]}")

Mixing the two dispersion curves genuinely decorrelates the band pair, a few
tenths of a percent at this modest scale, and it is non-stationary: the drop
from $\lambda_0$ is not a function of $|\lambda - \lambda_0|$ alone. The
matrix on the left is $\rho(\lambda_i, \lambda_j)$ at the sample pixel; the
curve on the right is one row of it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
im = axes[0].imshow(
    rho2[:, :, py, px], origin="lower", cmap="viridis", vmin=0.9, vmax=1.0
)
axes[0].set_title(r"$\rho(\lambda_i, \lambda_j)$ at the sample pixel")
axes[0].set_xlabel("channel j"), axes[0].set_ylabel("channel i")
fig.colorbar(im, ax=axes[0])
axes[1].plot(np.asarray(WLS), rho2[0, :, py, px], "o-")
axes[1].set_xlabel("wavelength [nm]")
axes[1].set_ylabel(r"$\rho(\lambda_0, \lambda)$")
axes[1].set_title("non-stationary decorrelation from the reference band")
plt.tight_layout()
plt.show()

## Effective channels and impropriety: mind the pixel

$N_\mathrm{eff}$ should now rise a little above its achromatic value of 1,
and {meth}`~physicaloptix.CrossBandMoments.impropriety` should stay small in
the dark zone: `fourier_dm_basis` returns cosine/sine pairs with equal rms
per frequency, which is exactly the paired, homogeneous basis the theory page
shows cancels the cross-band pseudo-covariance. The exact PSF core is a
different regime -- it is where every spatial frequency's cosine and sine
components interfere constructively by construction, not a translation-invariant
patch of speckle -- so the homogeneity argument does not apply there, and
evaluating impropriety at the core instead of the dark zone can overstate it
by an order of magnitude.

In [ ]:
n_eff2 = np.asarray(cbm2.n_eff())
eta2 = np.asarray(cbm2.impropriety())
off = ~np.eye(WLS.shape[0], dtype=bool)

print(
    f"n_eff in the dark zone: [{n_eff2[mask].min():.4f}, {n_eff2[mask].max():.4f}] "
    f"(ceiling m(m+3)/2 = {m * (m + 3) // 2})"
)
print(f"impropriety median in the dark zone: {np.median(eta2[off][:, mask]):.4f}")

core = np.unravel_index(
    np.abs(np.asarray(lin2.e_nom[0])).argmax(), lin2.e_nom.shape[1:]
)
print(
    f"impropriety at the exact PSF core {core}: {eta2[off][:, core[0], core[1]].mean():.4f}"
)

## The full spatio-spectral covariance

`joint_covariance` returns the complete `(band, pixel) x (band, pixel)`
object over a chosen pixel set; the per-pixel maps above are its
coincident-pixel diagonal. Checking that identity on two dark-zone pixels
confirms the two routes -- one einsum path over all pixels at once, a
second over a selected few -- agree exactly.

In [ ]:
small_mask = np.zeros(mask.shape, dtype=bool)
small_mask[py, px] = True
small_mask[ys[len(ys) // 3], xs[len(ys) // 3]] = True

joint = np.asarray(proc2.joint_covariance(jnp.asarray(small_mask)))
cov_map = np.asarray(cbm2.cov_map)
for k, (yy, xx) in enumerate(zip(*np.nonzero(small_mask))):
    diff = np.abs(joint[:, k, :, k] - cov_map[:, :, yy, xx]).max()
    print(f"pixel {k} at {(yy, xx)}: max diff between the two routes = {diff:.2e}")

## The achromatic-limit artifact, closed form vs code

[The theory page](../explanation/cross-band-statistics) derives that the
standard `lambda_scaled_channels` shortcut, which holds `E_nom` fixed across
channels by default, strictly decorrelates bands under coherent statistics:
with $c_i = \lambda_0/\lambda_i$, a heterodyne weight $K_1$, and a
speckle-only weight $K_2$ at the reference band,

$$
\rho_{ij} = \frac{c_i c_j K_2 + K_1}{\sqrt{(c_i^2 K_2 + K_1)(c_j^2 K_2 + K_1)}}.
$$

We rebuild a monochromatic linearization at $\lambda_0$, stack it two ways --
the frozen default and the `scale_e_nom=True` fix -- and compare the coded
$\rho$ against this closed form evaluated from the monochromatic `moments()`
output at the same pixel.

In [ ]:
lin0 = linearize(path, flat, basis, wavelength_nm=WL0)
per_mode_rms = jnp.full((basis.n_modes,), 5.0 / jnp.sqrt(basis.n_modes))

e_frozen, g_stack = lambda_scaled_channels(lin0.e_nom, lin0.G, WL0, WLS)
e_scaled, _ = lambda_scaled_channels(lin0.e_nom, lin0.G, WL0, WLS, scale_e_nom=True)

kwargs = dict(
    input_energy=lin0.input_energy,
    pixel_scale_lod=lin0.pixel_scale_lod,
    coherent=True,
    wavelengths_nm=WLS,
)
proc_frozen = SpeckleProcess(e_frozen, g_stack, per_mode_rms, 1e-3, **kwargs)
proc_scaled = SpeckleProcess(e_scaled, g_stack, per_mode_rms, 1e-3, **kwargs)
rho_f = np.asarray(proc_frozen.cross_band_moments().correlation())
rho_s = np.asarray(proc_scaled.cross_band_moments().correlation())

mono = SpeckleProcess(
    lin0.e_nom,
    lin0.G,
    per_mode_rms,
    1e-3,
    input_energy=lin0.input_energy,
    pixel_scale_lod=lin0.pixel_scale_lod,
    coherent=True,
)
mm = mono.moments()
K1 = 4.0 * np.abs(np.asarray(lin0.e_nom)) ** 2 * np.asarray(mm.var_x_map)
K2 = np.asarray(mm.gamma_map) ** 2 + np.abs(np.asarray(mm.p_map)) ** 2
c = WL0 / np.asarray(WLS)
pred = (c[0] * c[-1] * K2[py, px] + K1[py, px]) / np.sqrt(
    (c[0] ** 2 * K2[py, px] + K1[py, px]) * (c[-1] ** 2 * K2[py, px] + K1[py, px])
)

print(f"frozen E_nom:  coded rho(lam_0, lam_last) = {rho_f[0, -1, py, px]:.6f}")
print(f"closed form:   predicted rho(lam_0, lam_last) = {pred:.6f}")
print(f"scale_e_nom=True: coded rho(lam_0, lam_last) = {rho_s[0, -1, py, px]:.6f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(np.asarray(WLS), rho_f[0, :, py, px], "o-", label="frozen E_nom (default)")
ax.plot(np.asarray(WLS), rho_s[0, :, py, px], "s--", label="scale_e_nom=True")
ax.set_xlabel("wavelength [nm]")
ax.set_ylabel(r"$\rho(\lambda_0, \lambda)$")
ax.set_title("the frozen-E_nom achromatic artifact, and its fix")
ax.legend()
plt.tight_layout()
plt.show()

The closed form and the coded output agree to machine precision, and
`scale_e_nom=True` restores $\rho \equiv 1$ exactly, confirming the fix is
not just qualitative.

## Synthesis

- `linearize(..., wavelengths_nm=...)` builds the full chromatic `(E_nom, G)`
  stack in one call, with an optional `dispersion` table standing in for the
  default OPD law -- this is how a genuinely chromatic element such as a
  coating enters the model.
- A single shared dispersion curve cannot decorrelate bands, whatever its
  shape; two curves mixed across the mode set can, and the resulting
  $\rho(\lambda_i, \lambda_j)$ is non-stationary, as the closed form predicts.
- `n_eff` and `impropriety` are pixel-dependent diagnostics: a paired,
  homogeneous mode basis keeps impropriety small in the dark zone, but not at
  the PSF core, where the same-frequency cancellation the theory page relies
  on does not hold.
- `joint_covariance` and `cross_band_moments` are two views of the same
  object, verified to agree on their shared coincident-pixel diagonal.
- The frozen-`E_nom` lambda-scaling shortcut measurably decorrelates bands
  under coherent statistics, exactly as the closed form predicts, and
  `scale_e_nom=True` removes the artifact.

See [the cross-band statistics page](../explanation/cross-band-statistics)
for the full derivations behind every number here, and
[notebook 6](06_The_Speckle_Layer_in_Code) for the monochromatic
`linearize` / `stats` / `SpeckleProcess` API this one builds on.